> ## SOLUTION / ANSWER KEY &mdash; Lab 5.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-plan-and-execute.ipynb`](../lab-10-plan-and-execute.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.10 &mdash; Plan-and-Execute (vs ReAct)

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Write a planner that drafts all steps up front
- Write an executor that runs each step, feeding results forward
- Contrast plan-and-execute with step-by-step ReAct

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 5 slides &mdash; Planning patterns at a glance](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-05-10"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
**ReAct** decides one step at a time. **Plan-and-execute** drafts the **whole plan first**, then
runs it &mdash; better for long, structured tasks. Here a `planner(goal)` returns a list of
`{tool, input}` steps and an `executor` runs them, substituting the previous result into the next
step. (The optional cell lets a real LLM write the plan.)

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)
KNOWLEDGE = {"population of metropolis": "120000"}
TOOLS = {
    "lookup":     {"fn": lambda k: KNOWLEDGE.get(k.lower().strip(), "unknown")},
    "calculator": {"fn": safe_calc},
}
print("tools ready:", list(TOOLS))

## Your Turn
Write the two-step plan, then the executor that substitutes the previous RESULT into each step.

In [ ]:
def planner(goal):
    # draft ALL steps up front (a fixed plan for: twice the population of Metropolis)
    return [
        {"tool": "lookup", "input": "population of metropolis"},
        {"tool": "calculator", "input": "RESULT*2"},
    ]

def executor(plan):
    result, trace = None, []
    for step in plan:
        arg = step["input"].replace("RESULT", str(result)) if result is not None else step["input"]
        result = TOOLS[step["tool"]]["fn"](arg)
        trace.append((step["tool"], result))
    return result, trace

try:
    plan = planner('twice the population of metropolis')
    final, trace = executor(plan)
    print('plan:', plan)
    print('trace:', trace)
    print('final:', final)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("plan has two steps", lambda: len(planner("x")) == 2)
expect_true("first step uses lookup", lambda: planner("x")[0]["tool"] == "lookup")
expect_true("second step uses calculator", lambda: planner("x")[1]["tool"] == "calculator")
expect_true("executing the plan yields twice the population", lambda: executor(planner("x"))[0] == 240000)
expect_true("the trace records both tool runs", lambda: [t for t, _ in executor(planner("x"))[1]] == ["lookup", "calculator"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; swap the rule-based policy for a REAL LLM (not graded)
Let a REAL LLM WRITE the plan; your executor above still runs the tools deterministically. Safe to skip &mdash; it needs a local **Ollama** (`pip install langchain-ollama`, then
`ollama run llama3.2:1b`) or a **Groq** key (`pip install langchain-groq`, `GROQ_API_KEY`). If
neither is present the cell prints a friendly note and moves on. **The graded steps above run on a
deterministic rule-based policy, so the lab always verifies offline.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    prompt = ("List the 2 steps (each as tool + input) to find twice the population of Metropolis. "
              "Tools available: lookup, calculator. Be terse.")
    print("LLM plan:\n", llm.invoke(prompt).content)
    print("A real LLM can author the plan; the executor pattern above is unchanged.")
except Exception as e:
    print("No local LLM available -- skipping (install langchain-ollama + `ollama run llama3.2:1b`,", type(e).__name__)
    print("or use Groq: `from langchain_groq import ChatGroq` with GROQ_API_KEY).")
    print("The rule-based planner above already produced and executed a correct plan offline.")

---
### Key takeaway
Plan-and-execute front-loads the thinking; ReAct interleaves it. Same tools, different control pattern &mdash; pick the one that fits the task's shape.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>